# NBME DeBERTa-v3-large Kaggle Inference

這份 notebook 是給 Kaggle submission 用的，只做 inference，不重新訓練。

需要 add input：
- Kaggle 官方資料：`nbme-score-clinical-patient-notes`
- 你訓練輸出的模型資料夾，裡面要有 `config.pth`、`tokenizer/`、`oof_df.pkl`、以及 5 個 fold 的 `.pth` 權重

Internet 可以關閉。最後會輸出 `/kaggle/working/submission.csv`。

In [27]:
import os
from pathlib import Path

# Kaggle 官方競賽資料。Add data 後通常就是這個路徑。
COMP_DIR = Path('/kaggle/input/competitions/nbme-score-clinical-patient-notes')

# TODO: 改成你上傳模型權重的 Kaggle Dataset 路徑。
# 這個資料夾內應包含：config.pth, tokenizer/, oof_df.pkl,
# microsoft-deberta-v3-large_fold0_best.pth ... fold4_best.pth
MODEL_DIR = Path('/kaggle/input/datasets/shuruc/deberta-v3-large-v2')

OUTPUT_DIR = Path('/kaggle/working')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('COMP_DIR:', COMP_DIR)
print('MODEL_DIR:', MODEL_DIR)
print('MODEL_DIR files:')
if MODEL_DIR.exists():
    for p in sorted(MODEL_DIR.iterdir()):
        print(' -', p.name)
else:
    print('MODEL_DIR does not exist. Please change MODEL_DIR to your Kaggle input path.')

COMP_DIR: /kaggle/input/competitions/nbme-score-clinical-patient-notes
MODEL_DIR: /kaggle/input/datasets/shuruc/deberta-v3-large-v2
MODEL_DIR files:
 - added_tokens.json
 - config.pth
 - microsoft-deberta-v3-large_fold0_best.pth
 - microsoft-deberta-v3-large_fold1_best.pth
 - microsoft-deberta-v3-large_fold2_best.pth
 - microsoft-deberta-v3-large_fold3_best.pth
 - microsoft-deberta-v3-large_fold4_best.pth
 - oof_df.pkl
 - special_tokens_map.json
 - spm.model
 - tokenizer.json
 - tokenizer_config.json


In [28]:
class CFG:
    model = 'microsoft/deberta-v3-large'
    config_path = MODEL_DIR / 'config.pth'
    tokenizer_dir = MODEL_DIR
    batch_size = 8
    fc_dropout = 0.2
    max_len = None
    seed = 42
    num_workers = 0
    n_fold = 5
    trn_fold = [0, 1, 2, 3, 4]
    # 如果沒有 oof_df.pkl 可以手動固定 threshold，例如 0.50
    default_threshold = 0.50

In [29]:
import gc
import re
import ast
import random
import itertools
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import f1_score

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

import transformers
from transformers import AutoModel, AutoConfig, DebertaV2TokenizerFast

os.environ['TOKENIZERS_PARALLELISM'] = 'true'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('transformers:', transformers.__version__)
print('torch:', torch.__version__)
print('device:', device)

transformers: 5.0.0
torch: 2.10.0+cu128
device: cuda


In [30]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(CFG.seed)

In [31]:
def process_feature_text(text):
    text = re.sub('I-year', '1-year', str(text))
    text = re.sub('-OR-', ' or ', text)
    text = re.sub('-', ' ', text)
    return text


def micro_f1(preds, truths):
    preds = np.concatenate(preds)
    truths = np.concatenate(truths)
    return f1_score(truths, preds)


def spans_to_binary(spans, length=None):
    length = np.max(spans) if length is None else length
    binary = np.zeros(length)
    for start, end in spans:
        binary[start:end] = 1
    return binary


def span_micro_f1(preds, truths):
    bin_preds = []
    bin_truths = []
    for pred, truth in zip(preds, truths):
        if not len(pred) and not len(truth):
            continue
        length = max(np.max(pred) if len(pred) else 0,
                     np.max(truth) if len(truth) else 0)
        bin_preds.append(spans_to_binary(pred, length))
        bin_truths.append(spans_to_binary(truth, length))
    return micro_f1(bin_preds, bin_truths)


def create_labels_for_scoring(df):
    truths = []
    for char_spans in df['char_spans'].values:
        truths.append([[start, end] for start, end in char_spans])
    return truths


def get_char_probs(texts, predictions, tokenizer):
    results = [np.zeros(len(t)) for t in texts]
    for i, (text, prediction) in enumerate(zip(texts, predictions)):
        encoded = tokenizer(text, add_special_tokens=True, return_offsets_mapping=True)
        for offset_mapping, pred in zip(encoded['offset_mapping'], prediction):
            start, end = offset_mapping
            results[i][start:end] = pred
    return results


def get_results(char_probs, th=0.5):
    # Kaggle location format uses character start and exclusive end.
    results = []
    for char_prob in char_probs:
        idx = np.where(char_prob >= th)[0]
        groups = [list(g) for _, g in itertools.groupby(
            idx, key=lambda n, c=itertools.count(): n - next(c))]
        spans = [f'{min(g)} {max(g) + 1}' for g in groups]
        results.append(';'.join(spans))
    return results
/kaggle/input/datasets/shuruc/deberta-v3-large-v2/microsoft-deberta-v3-large_fold0_best.pth

def get_predictions(results):
    predictions = []
    for result in results:
        prediction = []
        if result != '':
            for loc in [s.split() for s in result.split(';')]:
                prediction.append([int(loc[0]), int(loc[1])])
        predictions.append(prediction)
    return predictions


def get_score(y_true, y_pred):
    return span_micro_f1(y_pred, y_true)

In [32]:
tokenizer = DebertaV2TokenizerFast.from_pretrained(CFG.tokenizer_dir)
CFG.tokenizer = tokenizer
print('Tokenizer loaded from:', CFG.tokenizer_dir)

Tokenizer loaded from: /kaggle/input/datasets/shuruc/deberta-v3-large-v2


In [33]:
oof_path = MODEL_DIR / 'oof_df.pkl'
best_th = CFG.default_threshold

if oof_path.exists():
    oof = pd.read_pickle(oof_path)
    pred_cols = [c for c in oof.columns if isinstance(c, int)]
    CFG.max_len = len(pred_cols)
    print('oof.shape:', oof.shape)
    print('max_len from oof_df:', CFG.max_len)

    truths = create_labels_for_scoring(oof)
    char_probs = get_char_probs(
        oof['pn_history'].values,
        oof[[i for i in range(CFG.max_len)]].values,
        CFG.tokenizer,
    )

    best_score = -1
    print('Threshold tuning:')
    for th in np.arange(0.45, 0.56, 0.01):
        th = np.round(th, 2)
        results = get_results(char_probs, th=th)
        preds = get_predictions(results)
        score = get_score(truths, preds)
        if score > best_score:
            best_score = score
            best_th = float(th)
        print(f'  th={th:.2f} score={score:.5f}')
    print(f'best_th={best_th:.2f} best_score={best_score:.5f}')
else:
    # Fallback: infer max_len from config/training convention.
    CFG.max_len = 325
    print('oof_df.pkl not found. Use default threshold:', best_th)
    print('max_len fallback:', CFG.max_len)

oof.shape: (14300, 335)
max_len from oof_df: 325
Threshold tuning:
  th=0.45 score=0.85171
  th=0.46 score=0.85162
  th=0.47 score=0.85174
  th=0.48 score=0.85187
  th=0.49 score=0.85188
  th=0.50 score=0.85195
  th=0.51 score=0.85206
  th=0.52 score=0.85184
  th=0.53 score=0.85182
  th=0.54 score=0.85193
  th=0.55 score=0.85185
  th=0.56 score=0.85178
best_th=0.51 best_score=0.85206


In [34]:
test = pd.read_csv(COMP_DIR / 'test.csv')
features = pd.read_csv(COMP_DIR / 'features.csv')
patient_notes = pd.read_csv(COMP_DIR / 'patient_notes.csv')
submission = pd.read_csv(COMP_DIR / 'sample_submission.csv')

# Public test.csv is tiny sample data; during Kaggle scoring it is replaced by hidden test rows.
test = test.merge(features, on=['feature_num', 'case_num'], how='left')
test = test.merge(patient_notes, on=['pn_num', 'case_num'], how='left')
test['feature_text'] = test['feature_text'].apply(process_feature_text)

test = test.sort_values('id').reset_index(drop=True)
submission = submission.sort_values('id').reset_index(drop=True)

print('test.shape:', test.shape)
print('submission.shape:', submission.shape)
display(test.head())

test.shape: (5, 6)
submission.shape: (5, 2)


,id,case_num,pn_num,feature_num,feature_text,pn_history
0,00016_000,0,16,0,Family history of MI or Family history of myoc...,HPI: 17yo M presents with palpitations. Patien...
1,00016_001,0,16,1,Family history of thyroid disorder,HPI: 17yo M presents with palpitations. Patien...
2,00016_002,0,16,2,Chest pressure,HPI: 17yo M presents with palpitations. Patien...
3,00016_003,0,16,3,Intermittent symptoms,HPI: 17yo M presents with palpitations. Patien...
4,00016_004,0,16,4,Lightheaded,HPI: 17yo M presents with palpitations. Patien...


In [35]:
def prepare_input(cfg, text, feature_text):
    inputs = cfg.tokenizer(
        text,
        feature_text,
        add_special_tokens=True,
        max_length=CFG.max_len,
        padding='max_length',
        truncation=True,
        return_offsets_mapping=False,
    )
    for k, v in inputs.items():
        inputs[k] = torch.tensor(v, dtype=torch.long)
    return inputs


class TestDataset(Dataset):
    def __init__(self, cfg, df):
        self.cfg = cfg
        self.feature_texts = df['feature_text'].values
        self.pn_historys = df['pn_history'].values

    def __len__(self):
        return len(self.feature_texts)

    def __getitem__(self, item):
        return prepare_input(self.cfg, self.pn_historys[item], self.feature_texts[item])

In [36]:
class CustomModel(nn.Module):
    def __init__(self, cfg, config_path=None, pretrained=False):
        super().__init__()
        self.cfg = cfg
        if config_path is None:
            self.config = AutoConfig.from_pretrained(cfg.model, output_hidden_states=True)
        else:
            self.config = torch.load(config_path, map_location='cpu', weights_only=False)
        
    
        # Compatibility for configs saved with older transformers versions.
        # Newer transformers expects these internal config fields when building from config.
        if not hasattr(self.config, 'dtype'):
            self.config.dtype = torch.float32
        if not hasattr(self.config, 'torch_dtype') or self.config.torch_dtype is None:
            self.config.torch_dtype = torch.float32
        if not hasattr(self.config, '_attn_implementation_internal'):
            self.config._attn_implementation_internal = None
        if not hasattr(self.config, '_experts_implementation_internal'):
            self.config._experts_implementation_internal = None
        if not hasattr(self.config, '_output_attentions'):
            self.config._output_attentions = False
        if not hasattr(self.config, '_output_hidden_states'):
            self.config._output_hidden_states = True
        if not hasattr(self.config, 'return_dict'):
            self.config.return_dict = True

        if pretrained:
            self.model = AutoModel.from_pretrained(cfg.model, config=self.config)
        else:
            self.model = AutoModel.from_config(self.config)

        self.fc_dropout = nn.Dropout(cfg.fc_dropout)
        self.fc = nn.Linear(self.config.hidden_size, 1)

    def feature(self, inputs):
        outputs = self.model(**inputs)
        return outputs[0]

    def forward(self, inputs):
        feature = self.feature(inputs)
        return self.fc(self.fc_dropout(feature.float()))

In [37]:
def inference_fn(test_loader, model, device):
    preds = []
    model.eval()
    model.to(device)

    for inputs in tqdm(test_loader, total=len(test_loader)):
        for k, v in inputs.items():
            inputs[k] = v.to(device)
        with torch.no_grad():
            y_preds = model(inputs)
        preds.append(y_preds.sigmoid().cpu().numpy())

    return np.concatenate(preds)

In [38]:
test_dataset = TestDataset(CFG, test)
test_loader = DataLoader(
    test_dataset,
    batch_size=CFG.batch_size,
    shuffle=False,
    num_workers=CFG.num_workers,
    pin_memory=True,
    drop_last=False,
)

all_char_probs = []

for fold in CFG.trn_fold:
    print(f'========== fold {fold} inference ==========')
    weight_path = MODEL_DIR / f"{CFG.model.replace('/', '-')}_fold{fold}_best.pth"
    if not weight_path.exists():
        raise FileNotFoundError(f'Missing weight file: {weight_path}')

    model = CustomModel(CFG, config_path=CFG.config_path, pretrained=False)
    state = torch.load(weight_path, map_location='cpu', weights_only=False)
    model.load_state_dict(state['model'])

    prediction = inference_fn(test_loader, model, device)
    prediction = prediction.reshape((len(test), CFG.max_len))
    char_probs = get_char_probs(test['pn_history'].values, prediction, CFG.tokenizer)
    all_char_probs.append(char_probs)

    del model, state, prediction, char_probs
    gc.collect()
    torch.cuda.empty_cache()

# Average folds per sample. This handles variable text lengths safely.
final_char_probs = [
    np.mean([fold_probs[i] for fold_probs in all_char_probs], axis=0)
    for i in range(len(test))
]
print('Ensemble done:', len(all_char_probs), 'folds')

========== fold 0 inference ==========


  0%|          | 0/1 [00:00<?, ?it/s]

========== fold 1 inference ==========


  0%|          | 0/1 [00:00<?, ?it/s]

========== fold 2 inference ==========


...

========== fold 3 inference ==========


  0%|          | 0/1 [00:00<?, ?it/s]

========== fold 4 inference ==========


  0%|          | 0/1 [00:00<?, ?it/s]

Ensemble done: 5 folds


In [39]:
results = get_results(final_char_probs, th=best_th)

submission = test[['id']].copy()
submission['location'] = results
submission = submission.sort_values('id').reset_index(drop=True)

sub_path = OUTPUT_DIR / 'submission.csv'
submission.to_csv(sub_path, index=False)

print('threshold:', best_th)
print('non-empty predictions:', (submission['location'] != '').sum(), '/', len(submission))
print('Saved:', sub_path)
display(submission.head(20))

threshold: 0.51
non-empty predictions: 5 / 5
Saved: /kaggle/working/submission.csv


,id,location
0,00016_000,695 724
1,00016_001,667 693
2,00016_002,202 217
3,00016_003,69 91
4,00016_004,221 258


In [40]:
check = pd.read_csv('/kaggle/working/submission.csv')
print(check.shape)
print(check.columns.tolist())
display(check.head())

(5, 2)
['id', 'location']


,id,location
0,00016_000,695 724
1,00016_001,667 693
2,00016_002,202 217
3,00016_003,69 91
4,00016_004,221 258
